<a href="https://colab.research.google.com/github/blankqspace/homework_compling_course/blob/main/llm_intro_homework_Artamonova.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашнее задание № 4. Языковые модели

## Задание 1 (8 баллов).

В семинаре для генерации мы использовали предположение маркова и считали, что слово зависит только от 1 предыдущего слова. Но ничто нам не мешает попробовать увеличить размер окна и учитывать два или даже три прошлых слова. Для них мы еще сможем собрать достаточно статистик и, логично предположить, что качество сгенерированного текста должно вырасти.

Попробуйте сделать языковую модель, которая будет учитывать два предыдущих слова при генерации текста.
Сгенерируйте несколько текстов (3-5) и расчитайте перплексию получившейся модели.
Можно использовать данные из семинара или любые другие (можно брать только часть текста, если считается слишком долго). Перплексию рассчитывайте на 10-50 отложенных предложениях (они не должны использоваться при сборе статистик).


Подсказки:  
    - нужно будет добавить еще один тэг \<start>  
    - можете использовать тот же подход с матрицей вероятностей, но по строкам хронить биграмы, а по колонкам униграммы
    - тексты должны быть очень похожи на нормальные (если у вас получается рандомная каша, вы что-то делаете не так)
    - у вас будут словари с индексами биграммов и униграммов, не перепутайте их при переводе индекса в слово - словарь биграммов будет больше словаря униграммов и все индексы из униграммного словаря будут формально подходить для словаря биграммов (не будет ошибки при id2bigram[unigram_id]), но маппинг при этом будет совершенно неправильным

In [9]:
dvach = open('2ch_corpus.txt').read()

In [10]:
!pip install nltk

In [8]:
!pip install razdel

In [11]:
from scipy.sparse import lil_matrix, csr_matrix, csc_matrix
from string import punctuation
from razdel import sentenize
from razdel import tokenize as razdel_tokenize
from collections import Counter
import numpy as np
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [12]:
def normalize(text):
    normalized_text = [word.text.strip(punctuation) for word \
                                                            in razdel_tokenize(text)]
    normalized_text = [word.lower() for word in normalized_text if word and len(word) < 20 ]
    return normalized_text


norm_dvach = normalize(dvach)
vocab_dvach = Counter(norm_dvach)

print("Длина корпуса текстов c Двача в токенах -", len(norm_dvach))
print("Топ-10 самых частотных слов из 2ch:", vocab_dvach.most_common(10))

Длина корпуса текстов c Двача в токенах - 1858941
Топ-10 самых частотных слов из 2ch: [('и', 55892), ('в', 48853), ('не', 46602), ('на', 29660), ('что', 26668), ('я', 21734), ('а', 21310), ('с', 21080), ('это', 17727), ('ты', 15469)]


In [13]:
import numpy as np
from scipy.sparse import lil_matrix, csc_matrix
from collections import Counter
import re
from nltk.tokenize import sent_tokenize

def normalize(text):
    normalized_text = [word.text.strip(punctuation) for word \
                                                            in razdel_tokenize(text)]
    normalized_text = [word.lower() for word in normalized_text if word and len(word) < 20 ]
    return normalized_text

def ngrammer(tokens, n=2):
    ngrams = []
    for i in range(0,len(tokens)-n+1):
        ngrams.append(' '.join(tokens[i:i+n]))
    return ngrams

def apply_temperature(probas, temperature):
    log_probas = np.log(np.maximum(probas, 1e-10))
    adjusted_log_probas = log_probas / temperature
    exp_probas = np.exp(adjusted_log_probas)
    adjusted_probabilities = exp_probas / np.sum(exp_probas)
    return adjusted_probabilities

sentences_dvach = [['<start1>', '<start2>'] + normalize(text) + ['<end>'] for text in sent_tokenize(dvach[:500000])]

unigrams_dvach = Counter()
bigrams_dvach = Counter()
trigrams_dvach = Counter()

for sentence in sentences_dvach:
    unigrams_dvach.update(sentence)
    bigrams_dvach.update(ngrammer(sentence, 2))
    trigrams_dvach.update(ngrammer(sentence, 3))

# Строки - биграммы (два предыдущих слова), столбцы - следующие слова
id2word_dvach = list(unigrams_dvach)
word2id_dvach = {word: i for i, word in enumerate(id2word_dvach)}

id2bigram_dvach = list(bigrams_dvach)
bigram2id_dvach = {bigram: i for i, bigram in enumerate(id2bigram_dvach)}

matrix_dvach = lil_matrix((len(bigrams_dvach), len(unigrams_dvach)))

for trigram in trigrams_dvach:
    words = trigram.split()
    if len(words) == 3:
        word1, word2, word3 = words
        bigram = f"{word1} {word2}"
        if bigram in bigram2id_dvach and word3 in word2id_dvach:
            bigram_id = bigram2id_dvach[bigram]
            word3_id = word2id_dvach[word3]
            matrix_dvach[bigram_id, word3_id] = (trigrams_dvach[trigram] /
                                                bigrams_dvach[bigram])

matrix_dvach = csc_matrix(matrix_dvach)

def generate_two_prev(matrix, id2word, word2id, id2bigram, bigram2id, n=100,
                     start1='<start1>', start2='<start2>', temperature=1.):
    text = []
    current_bigram = f"{start1} {start2}"
    current_bigram_id = bigram2id[current_bigram]

    for i in range(n):
        chosen = np.random.choice(matrix.shape[1],
                                  p=apply_temperature(matrix[current_bigram_id].toarray()[0],
                                                      temperature=temperature))
        next_word = id2word[chosen]
        text.append(next_word)

        if next_word == '<end>':
            current_bigram = f"<start1> <start2>"
            current_bigram_id = bigram2id[current_bigram]
        else:
            prev_words = current_bigram.split()
            new_bigram = f"{prev_words[1]} {next_word}"
            if new_bigram in bigram2id:
                current_bigram_id = bigram2id[new_bigram]
                current_bigram = new_bigram
            else:
                current_bigram = f"<start1> <start2>"
                current_bigram_id = bigram2id[current_bigram]

    return ' '.join(text)

for i in range(3):
    text = generate_two_prev(matrix_dvach, id2word_dvach, word2id_dvach,
                           id2bigram_dvach, bigram2id_dvach, n=50, temperature=0.8)
    print(f"Текст {i+1}: {text}")

Текст 1: где вы ведите условие того что все чертежи давно выложены на стол и давай страдать хуйней в школьных тетрадях описывал всякие фэнтезийные сеттинги рисовал карты лица героев в профиль мечи разные легендарные доспехи и прочую чушь <end> сам на пик посмотри откуда 2 картинка пошла <end> 2016 заливать картинки со
Текст 2: они регистрировались на рейс до сочи <end> по твоей логике перед тем как учить сишку нужно сначала изучить как работает радиация <end> как это ты оперируешь понятиями с которыми я не знаю как выйти но мне смущает в саду … в этот разговор вставить <end> они всегда ссылаются на это
Текст 3: каждый раз ловит инфаркт и помирает когда пытается реснутся <end> 2 <end> разница 1,5/2 градуса от реальной температуры <end> почему когда я был бы я не могу понять к чему эти лишние пояснения описывается хиккан который явно не был закрыт в сейфе на 10 замком о том что я говорю


In [14]:
def compute_joint_proba_markov_assumption(text, word_counts, bigram_counts):
    prob = 0
    tokens = normalize(text)
    for ngram in ngrammer(['<start>'] + tokens + ['<end>']):
        word1, word2 = ngram.split()
        if word1 in word_counts and ngram in bigram_counts:
            prob += np.log(bigram_counts[ngram]/word_counts[word1])
        else:
            prob += np.log(2e-5)

    return prob, len(tokens)


def compute_joint_proba_two_prev(text, unigrams, bigrams, trigrams):
    prob = 0
    tokens = normalize(text)
    extended_tokens = ['<start1>', '<start2>'] + tokens + ['<end>']

    for i in range(2, len(extended_tokens)):
        word1, word2, word3 = extended_tokens[i-2], extended_tokens[i-1], extended_tokens[i]
        bigram = f"{word1} {word2}"
        trigram = f"{word1} {word2} {word3}"

        if bigram in bigrams and trigram in trigrams:
            prob += np.log(trigrams[trigram] / bigrams[bigram])
        else:
            prob += np.log(2e-5)

    return prob, len(tokens)

def perplexity(logp, N):
    return np.exp((-1/N) * logp)

In [15]:
phrases = "А есть видос с пикчи? Хочу посмотреть, как у людей бомбит от скримеров, лол. Пожалуйста. С меня нефть говорят он столыпина слил с потрохами.А есть видос с пикчи? А я почему раньше часто выпиливался? Потому что у меня портвешка не было. А теперь я добреть начну. И какую-нибудь няшу заведу. Сегодня собирали совещание, по поводу \"хакера Двача\", предупредили ничего не качать, а если с компьютером произойдёт что-то неладное - сразу идти к сисадмину."

In [16]:
perplexity(*compute_joint_proba_markov_assumption(phrases, unigrams_dvach, bigrams_dvach))

np.float64(7105.664310981538)

In [17]:
perplexity(*compute_joint_proba_two_prev(phrases, unigrams_dvach, bigrams_dvach, trigrams_dvach))

np.float64(34502.36088440728)

## Задание № 2* (2 балла).

Измените функцию generate_with_beam_search так, чтобы она работала с моделью, которая учитывает два предыдущих слова.
Сравните получаемый результат с первым заданием.
Также попробуйте начинать генерацию не с нуля (подавая \<start> \<start>), а с какого-то промпта. Но помните, что учитываться будут только два последних слова, так что не делайте длинные промпты.

In [18]:
class Beam:
    def __init__(self, sequence: list, score: float):
        self.sequence: list = sequence
        self.score: float = score

In [44]:
def generate_with_beam_search_two_prev(matrix, id2word, word2id, id2bigram, bigram2id,
                                      n=100, max_beams=5, start1='<start1>', start2='<start2>',
                                      prompt='мне не нравится'):
    if prompt:
        prompt_words = prompt.split()[-2:]
        if len(prompt_words) == 2:
            start1, start2 = prompt_words[0], prompt_words[1]
            initial_sequence = [start1, start2]
        elif len(prompt_words) == 1:
            start1, start2 = '<start1>', prompt_words[0]
            initial_sequence = [start1, start2]
        else:
            initial_sequence = [start1, start2]
        initial_score = 0.0
    else:
        initial_sequence = [start1, start2]
        initial_score = 0.0
    initial_node = Beam(sequence=initial_sequence, score=initial_score)
    beams = [initial_node]

    for i in range(n):
        new_beams = []

        for beam in beams:
            if beam.sequence and beam.sequence[-1] == '<end>':
                new_beams.append(beam)
                continue

            if len(beam.sequence) >= 2:
                last_two_words = beam.sequence[-2:]
                bigram = f"{last_two_words[0]} {last_two_words[1]}"
            else:
                if len(beam.sequence) == 1:
                    bigram = f"<start1> {beam.sequence[0]}"
                else:
                    bigram = "<start1> <start2>"

            if bigram in bigram2id:
                bigram_id = bigram2id[bigram]
                probas = matrix[bigram_id].toarray()[0]

                top_idxs = probas.argsort()[:-(max_beams+1):-1]

                for top_id in top_idxs:
                    if not probas[top_id]:
                        break

                    next_word = id2word[top_id]
                    new_sequence = beam.sequence + [next_word]

                    new_score = beam.score + np.log(probas[top_id] + 1e-10)
                    normalized_score = new_score / len(new_sequence)

                    new_beam = Beam(sequence=new_sequence, score=normalized_score)
                    new_beams.append(new_beam)
            else:
                new_beams.append(beam)

        if new_beams:
            beams = sorted(new_beams, key=lambda x: x.score, reverse=True)[:max_beams]
        else:
            break

    sorted_beams = sorted(beams, key=lambda x: x.score, reverse=True)
    return [" ".join(beam.sequence) for beam in sorted_beams]

In [47]:
generate_with_beam_search_two_prev(matrix_dvach, id2word_dvach, word2id_dvach, id2bigram_dvach, bigram2id_dvach, max_beams=10)

['не нравится пиздуй в хирургтческое отделение пиздошить себе аппендюк посмотри на ютубе как повар оливер вроде показывает из чего что вычел из твоей мамки свой хуйхохоха я давно закончил уже вуз <end>',
 'не нравится оформление диалогов по правилам написания художественного текста годные <end>',
 'не нравится я чувствую завтра не иду хоть что-то кажется мне странным <end>',
 'не нравится пиздуй в хирургтческое отделение пиздошить себе аппендюк посмотри на ютубе как повар оливер вроде показывает из чего что вычел из твоей вселенной <end>',
 'не нравится я чувствую завтра не иду хоть что-то приятное <end>',
 'не нравится гатари говно еще то <end>',
 'не нравится пиздуй в хирургтческое отделение пиздошить себе аппендюк посмотри на ютубе как повар оливер вроде показывает из чего что вычел чтобы получить это пиздец <end>',
 'не нравится его агрессивность <end>',
 'не нравится я чувствую в них да-да <end>',
 'не нравится пиздуй в хирургтческое отделение пиздошить себе аппендюк посмотри на ю